# 16 — Compatibility Report, Migration Shims, and Ecosystem Bridges

**Purpose:** The "I'm coming from another tool / my model has exotic plumbing"
workflow: `tl.compat.report` (pre-flight compatibility diagnosis), the `compat.from_*`
migration shims, the `lovely`/`torchshow` display shims, and the `tl.bridge` adapter
roster with its optional-dependency gating UX.

**Surfaces covered:**
- [ ] `tl.compat.report(model, x)` — `CompatReport` repr, `.rows` (`CompatRow`), `.to_markdown()`
- [ ] `tl.compat.from_fx(graph_module)` — FX-traced model migration
- [ ] `tl.compat.from_ilg(model, return_layers)` — IntermediateLayerGetter migration
- [ ] `tl.compat.from_torchextractor(model, layers)` — gated on `compat-shims` extra
- [ ] `tl.compat.from_timm` / `from_huggingface` / `from_sentence_transformers` — guarded
- [ ] `tl.compat.lovely` / `tl.compat.torchshow` — display shims
- [ ] `tl.bridge` — all 16 adapters enumerated; import-gating UX for missing deps
- [ ] `tl.bridge.captum` — `source_model`, `layer`, `attribute` (captum is installed here)
- [ ] `tl.bridge.profiler.execution_trace(trace, path)` — profiler-format export

## 1. Environment setup

In [ ]:
import pathlib
import sys
import warnings

# Pin imports to THIS checkout: the notebook dir (for _models.py) plus the repo
# root, so `import torchlens` audits the code this notebook ships with -- not a
# pip-installed copy from another checkout.
_NB_DIR = pathlib.Path.cwd()
if not (_NB_DIR / "_models.py").exists():
    _NB_DIR = next(
        p
        for p in [_NB_DIR / "notebooks" / "audit", *_NB_DIR.parents]
        if (p / "_models.py").exists()
    )
_REPO_ROOT = _NB_DIR.parents[1]
sys.path.insert(0, str(_NB_DIR))
sys.path.insert(0, str(_REPO_ROOT))

warnings.filterwarnings("ignore", category=DeprecationWarning)

import torch
import torchlens as tl

assert pathlib.Path(tl.__file__).is_relative_to(_REPO_ROOT), (
    f"torchlens imported from {tl.__file__} -- expected this checkout"
)
from _models import ZOO

print(f"torchlens version : {tl.__version__}")
print(f"torch version     : {torch.__version__}")

## 2. `tl.compat.report` — pre-flight compatibility diagnosis

The first thing to run on an unfamiliar model: an 18-row scan for wrappers, offload
hooks, quantization, compilation artifacts, and other trace hazards. Returns a
`CompatReport` with structured `CompatRow`s and a markdown renderer.

In [ ]:
model, x = ZOO["tiny_mlp"]()
rep = tl.compat.report(model, x)

print("type:", type(rep).__name__)
print()
print("First 3 rows (structured):")
for row in rep.rows[:3]:
    print(
        f"  {row.label:<32} status={row.status!r} severity={row.severity!r} detected={row.detected}"
    )
print()
non_pass = [r for r in rep.rows if r.status != "pass"]
print(f"rows: {len(rep.rows)} total, {len(non_pass)} non-pass on tiny_mlp")

In [ ]:
# The human-readable rendering
print(rep.to_markdown()[:900])
print("...")

## 3. Migration shims — `from_fx`, `from_ilg`, `from_torchextractor`

Each shim adapts a model-wrapping idiom from another tool into TorchLens terms.
`from_torchextractor` is gated behind the `compat-shims` extra — the error message is
part of the UX under audit.

In [ ]:
import torch.fx as fx

model, x = ZOO["tiny_mlp"]()

# from_fx: accepts an FX GraphModule
gm = fx.symbolic_trace(model)
fx_result = tl.compat.from_fx(gm)
print("from_fx ->", type(fx_result).__name__, "keys:", list(fx_result))
print("  schema:", fx_result.get("schema"))
print()

# from_ilg: torchvision IntermediateLayerGetter idiom -> Extractor
ilg = tl.compat.from_ilg(model, {"in_proj": "first", "out_proj": "last"})
print("from_ilg ->", type(ilg).__name__)
try:
    got = ilg(x)
    print(
        "  extractor(x) ->",
        type(got).__name__,
        {k: tuple(v.shape) for k, v in got.items()} if isinstance(got, dict) else "",
    )
except Exception as exc:
    print(f"  calling extractor -> {type(exc).__name__}: {str(exc)[:100]}")
print()

# from_torchextractor: gated on an extra -- show the gating message verbatim
try:
    tl.compat.from_torchextractor(model, ["in_proj"])
    print("from_torchextractor -> OK (extra installed)")
except ImportError as exc:
    print(f"from_torchextractor -> ImportError: {exc}")

In [ ]:
# from_timm / from_huggingface / from_sentence_transformers: guarded (no downloads)
for name, call in [
    ("from_timm", lambda: tl.compat.from_timm("resnet18", pretrained=False)),
    (
        "from_huggingface",
        lambda: tl.compat.from_huggingface(
            "hf-internal-testing/tiny-random-distilbert", local_files_only=True
        ),
    ),
]:
    try:
        obj = call()
        print(f"{name} -> {type(obj).__name__}")
    except ImportError as exc:
        print(f"{name} -> ImportError (dep gating): {str(exc)[:100]}")
    except Exception as exc:
        print(f"{name} -> {type(exc).__name__}: {str(exc)[:100]}")

## 4. Display shims — `compat.lovely` and `compat.torchshow`

Tiny adapters mirroring `lovely-tensors` (`lovely(t)` one-line tensor summary) and
`torchshow` (`show(t)`).

In [ ]:
t = torch.randn(2, 8)

try:
    print("compat.lovely.lovely(t) :", tl.compat.lovely.lovely(t))
except Exception as exc:
    print(f"lovely -> {type(exc).__name__}: {str(exc)[:110]}")

try:
    out = tl.compat.torchshow.show(t)
    print("compat.torchshow.show(t) ->", type(out).__name__)
except Exception as exc:
    print(f"torchshow.show -> {type(exc).__name__}: {str(exc)[:110]}")

## 5. The `tl.bridge` roster — 16 adapters and their gating UX

Import every adapter and record what a user sees: clean import, a clear
"install the extra" ImportError, or (worst case) a third-party package's own crash
leaking through raw.

In [ ]:
import importlib

adapters = list(tl.bridge.__all__)
print(f"tl.bridge.__all__ has {len(adapters)} adapters:\n")

for name in adapters:
    try:
        mod = importlib.import_module(f"torchlens.bridge.{name}")
        public = [
            n
            for n in dir(mod)
            if not n.startswith("_")
            and n not in {"Any", "annotations", "cast", "nn", "torch", "json", "Path", "join"}
        ]
        print(f"  {name:<18} OK       exports: {public[:5]}")
    except ImportError as exc:
        print(f"  {name:<18} GATED    {str(exc)[:80]}")
    except Exception as exc:
        print(f"  ⚠️ {name:<16} CRASH    {type(exc).__name__}: {str(exc)[:70]}")

## 6. `tl.bridge.captum` end-to-end (captum installed on this machine)

The adapter surfaces a traced model back to Captum: `source_model(trace)` returns the
original module, `layer(trace, site)` resolves a TorchLens site to the `nn.Module`
Captum wants, and `attribute(trace, method_obj, target)` drives a Captum method.

In [ ]:
try:
    import captum.attr

    from torchlens.bridge import captum as bcaptum

    model, x = ZOO["tiny_mlp"]()
    trace = tl.trace(model, x, intervention_ready=True)

    src_model = bcaptum.source_model(trace)
    print("source_model(trace) ->", type(src_model).__name__)

    relu_module = bcaptum.layer(trace, "relu_1_2")
    print("layer(trace, 'relu_1_2') ->", type(relu_module).__name__)

    sal_method = captum.attr.Saliency(src_model)
    try:
        res = bcaptum.attribute(trace, sal_method, 0)
        print(
            "attribute(trace, Saliency, 0) ->",
            type(res).__name__,
            tuple(res.shape) if hasattr(res, "shape") else "",
        )
    except Exception as exc:
        print(f"⚠️ attribute -> {type(exc).__name__}: {str(exc)[:130]}")
except ImportError:
    print("captum not installed -- section skipped (gating verified in §5)")

## 7. `tl.bridge.profiler` — execution-trace export

In [ ]:
import json as _json
import tempfile
import os

from torchlens.bridge import profiler as bprof

model, x = ZOO["tiny_mlp"]()
trace = tl.trace(model, x)

with tempfile.TemporaryDirectory() as td:
    out_path = os.path.join(td, "exec_trace.json")
    try:
        et = bprof.execution_trace(trace, out_path)
        print("execution_trace ->", type(et).__name__)
        print("  keys:", list(et)[:8] if isinstance(et, dict) else "")
        print("  file written:", os.path.exists(out_path), f"({os.path.getsize(out_path):,} B)")
    except Exception as exc:
        print(f"⚠️ execution_trace -> {type(exc).__name__}: {str(exc)[:130]}")

---

## ⚠️ GAPs / ergonomic smells

- **`tl.compat.report` emits stray stderr noise** — two
  `AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'` lines
  (protobuf probing) print on every call. The report itself is excellent; the probe
  should swallow its own diagnostics.
- **Broken third-party packages crash raw through the bridge roster** — an installed
  but import-broken dependency (nnsight on this machine) surfaces its own traceback
  through `import torchlens.bridge.nnsight` instead of a "dependency present but
  failing to import" message. Gating only catches the *absent* case.
- **Bridge adapter modules leak plumbing names** — `dir(bridge.captum)` shows `Any`,
  `annotations`, `cast`, `nn` alongside the real API; per-module `__all__` would fix
  tab-completion (same hygiene issue as `tl.facets`).
- **`from_torchextractor` vs `parquet` gating asymmetry** — compat shims name their
  extra (`install torchlens[compat-shims]`, good); some export/bridge paths raise raw
  ImportError with no install hint.
- The executed outputs of §5 record the live gating message for every adapter — any
  adapter whose message does not name its extra is an action item.